# 07 · Tracing and eval

To compare three frameworks, every run has to be measured the same way: the same definition of a token,
an LLM call, a tool call and a second. This notebook connects Strands to the shared harness in
`agent_contracts.harness`, measures runs of the research workflow from notebook 05, and scores them with
the shared scoring, which is how the capstone's `run_eval.py` will work.

**You will learn**

- what the shared harness measures, and where each number comes from
- how a Strands hook feeds model calls into it
- how to run a small, scored evaluation and write it in the shared results format

In [1]:
from agent_contracts.harness import RunMetrics, build_results, verify_model
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report
from strands_agents_impl.llm import make_model
from strands_agents_impl.research_flow import run_research, to_report

verify_model()  # stops with the command that fixes it if Ollama or the shared model is missing
model = make_model()  # the shared model, in Strands' form
questions = {q["id"]: q for q in load_questions()}  # the shared questions by id, each with its expected facts

## The shared definitions

| Number | Definition | Where it comes from |
|---|---|---|
| LLM calls | every model call in the run | the framework's callbacks, events or hooks |
| input and output tokens | summed over every model call | the counts Ollama reports with each response |
| tool calls | every attempted call, refused ones included | the shared tools record them (`research.tool_calls()`) |
| latency | wall clock for the whole question | `RunMetrics.measure()` |

Tool calls are counted inside the shared tools, so no framework can count them differently. Model calls
have to come from the framework, because only it sees them; in Strands that is a **hook**.

`HarnessHook`, from `strands_agents_impl/metrics.py`, is the whole bridge between Strands' hooks and the shared `RunMetrics`:

```python
class HarnessHook(HookProvider):
    def __init__(self, metrics: RunMetrics) -> None:
        self.metrics = metrics  # the shared counters for this run
        self._seen: dict[int, tuple[int, int, int]] = {}  # per agent: cycles, input, output already counted

    # Strands calls register_hooks once, when the hook is given to an Agent: subscribe to one event here.
    def register_hooks(self, registry: HookRegistry, **kwargs) -> None:
        registry.add_callback(AfterInvocationEvent, self.after_invocation)

    # Runs after every agent call (every agent("...")), once the agent's whole loop has finished.
    def after_invocation(self, event: AfterInvocationEvent) -> None:
        # Strands keeps running totals per agent: cycles (one model call each) and tokens, since it was made.
        m = event.agent.event_loop_metrics
        cycles = int(m.cycle_count)
        tokens_in, tokens_out = int(m.accumulated_usage.get("inputTokens", 0)), int(m.accumulated_usage.get("outputTokens", 0))
        # Subtract what was already counted for this agent, so a second call to it adds only its own share.
        seen_cycles, seen_in, seen_out = self._seen.get(id(event.agent), (0, 0, 0))
        self._seen[id(event.agent)] = (cycles, tokens_in, tokens_out)
        new_calls = cycles - seen_cycles
        if new_calls <= 0:
            return
        # The totals cannot be split per call, so all new tokens go on the first new call and the rest
        # are counted with zero tokens: the call count and the token total both come out right.
        self.metrics.add_llm_call(tokens_in - seen_in, tokens_out - seen_out)  # first call carries the tokens
        for _ in range(new_calls - 1):
            self.metrics.add_llm_call(0, 0)
```

Strands keeps running totals per agent (`agent.event_loop_metrics`) but updates the token counts only
after a model call's events have fired, so a hook that read them after every call would always be one call
behind. This one settles at the end of each agent invocation instead, which gives exact totals and call
counts. The research workflow's structured steps are single model calls made outside any agent
(notebook 05), so `structured_call` records those itself.

## One measured run

In [3]:
def run_question(qid: str) -> dict:
    """Run one question through the workflow, measured and scored the shared way."""
    q = questions[qid]
    research.reset()  # clear the shared tools' call record, so only this question's calls are counted
    metrics = RunMetrics()
    with metrics.measure():  # times the run
        state = run_research(qid, q["question"], model, metrics=metrics)  # metrics: where model calls are counted
    report = to_report(state)
    scored = score_report(report, q)  # compares the report with the question's expected facts
    # One result per question, in the shared format.
    return {"id": qid, "passed": scored["passed"], "scores": scored["scores"], "problems": scored["problems"],
            "report": report.model_dump(), "tool_calls": research.tool_calls(),
            "budget_exhausted": research.budget_exhausted(), **metrics.as_dict(), "error": None}

Now run one question, `Q07_deployment`, and print what came back: whether it passed, what the run cost,
and the answer with its scores.

In [4]:
item = run_question("Q07_deployment")

print("passed:         ", item["passed"])
print("model calls:    ", item["llm_calls"])
print("tokens in / out:", item["input_tokens"], "/", item["output_tokens"])
print("time:           ", f"{item['latency_s']:.0f} s")
print("tools called:   ", ", ".join(item["tool_calls"]) or "none")
print("answer:         ", item["report"]["answer"])
print("fact recall:    ", item["scores"]["fact_recall"], "(share of the expected facts the answer contains)")
print("facts missing:  ", ", ".join(item["scores"]["facts_missing"]) or "none")
print("no stale figure:", item["scores"]["contradiction_free"])
print("citations valid:", item["scores"]["citations_valid"])
print("problems:       ", "; ".join(item["problems"]) or "none")

passed:          False
model calls:     8
tokens in / out: 10870 / 461
time:            117 s
tools called:    web_search, web_search, web_search, web_search
answer:          Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025 [F04]. Earlier reports from 2023 indicated that TidePure units were operational in 9 provinces [F03].
fact recall:     1.0 (share of the expected facts the answer contains)
facts missing:   none
no stale figure: False
citations valid: True
problems:        repeats stale or wrong figures: ['9 provinces']


Q07 asks how many units Veltara has deployed and in how many provinces. The 2023 article still says
9 provinces; the current figure is 14. Mentioning 9 at all fails `contradiction_free`.

## A small eval

An evaluation is the same thing over a fixed set of cases, scored automatically, so you can re-run it after
every change and see whether things got better. These are the same questions the LangGraph and CrewAI
tutorials use: a calculation (Q03), a recall with a distractor article about another company (Q08), and the
Series B round (Q04).

In [5]:
# Run the other three questions with the same function and keep every result in one list.
items = [item]
for qid in ("Q03_revenue_growth", "Q08_recall", "Q04_series_b"):
    items.append(run_question(qid))
    print("finished", qid)

finished Q03_revenue_growth


finished Q08_recall


finished Q04_series_b


When the runs finish, print one row per question: whether it passed, how much of the expected facts it
recalled, and what it cost.

In [6]:
# One row per question: pass or fail, recall of the expected facts, and the cost of the run.
print(f"{'question':20} {'pass':5} {'recall':6} {'calls':5} {'tokens':>7} {'secs':>5}  tools")
for it in items:
    print(f"{it['id']:20} {str(it['passed']):5} {it['scores']['fact_recall']:6} {it['llm_calls']:5} "
          f"{it['input_tokens'] + it['output_tokens']:7} {it['latency_s']:5.0f}  {', '.join(it['tool_calls'])}")

passed = sum(it["passed"] for it in items)
print(f"\n{passed} of {len(items)} questions passed")

# For any question that failed a check, show what the check found and the answer that triggered it.
for it in items:
    if it["problems"]:
        print(f"\n{it['id']} failed a check")
        print("  problems:", "; ".join(it["problems"]))
        print("  answer:  ", it["report"]["answer"])

question             pass  recall calls  tokens  secs  tools
Q07_deployment       False    1.0     8   11331   117  web_search, web_search, web_search, web_search
Q03_revenue_growth   True     1.0     6    6199   123  web_search, web_search, calculator
Q08_recall           True     1.0     5    2978    71  web_search
Q04_series_b         True     1.0     5    3162    75  web_search

3 of 4 questions passed

Q07_deployment failed a check
  problems: repeats stale or wrong figures: ['9 provinces']
  answer:   Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025 [F04]. Earlier reports from 2023 indicated that TidePure units were operational in 9 provinces [F03].


When we ran it, three of four passed. Q07 failed: the answer gave the current 14 provinces, then added
that "earlier reports from 2023 indicated ... 9 provinces", citing the stale article. The critic passed
it; the plain-text check did not. LangGraph's Q01 can fail the same way (see notebook 05 there): a model
that knows which figure is current still mentions the old one, and a model acting as critic lets it
through.

Across the three tutorials, the same four questions with the same instructions and model:

| | LangGraph | CrewAI | Strands |
|---|---|---|---|
| passed | 4 of 4 | 4 of 4 | 3 of 4 |
| total tokens | 15,669 | 33,849 | 23,670 |
| total time | 439 s | 580 s | 386 s |
| web searches | 5 | 11 | 8 |

These are single runs of four questions on one laptop, so treat the differences as first signals, not a
verdict; the capstone evaluations run every question in every framework.

## The shared results format

`build_results` wraps the items in the format every framework writes, and computes the summary the same way
for all of them. The capstone's `run_eval.py` writes it to `results/research/strands.json`; this notebook
only prints it, so a tutorial run never overwrites real results.

In [7]:
from importlib.metadata import version

from agent_contracts.harness import load_settings

# Wrap the four results in the shared format; the framework's name and version are recorded in the file.
results = build_results("research", "strands", version("strands-agents"), load_settings().model, items)

summary = results["summary"]  # computed the same way for every framework
print("model:        ", results["model"])
print("passed:       ", f"{summary['passed']} of {summary['total']} questions")
print("total tokens: ", summary["total_tokens"])
print("total time:   ", f"{summary['total_latency_s']:.0f} s")
print("fields saved for each question:", ", ".join(sorted(results["questions"][0])))

model:         qwen3-agents
passed:        3 of 4 questions
total tokens:  23670
total time:    386 s
fields saved for each question: budget_exhausted, error, id, input_tokens, latency_s, llm_calls, output_tokens, passed, problems, report, scores, tool_calls


## Seeing inside a run

Numbers say *how much*; traces say *what happened*. In Strands the simplest trace is the conversation
itself, `agent.messages`, with every `toolUse` and `toolResult` (notebook 02), and the hooks used
throughout this tutorial can log any event in the loop. Strands also emits OpenTelemetry traces, which you
can send to a collector you run; nothing is sent anywhere unless you configure an exporter.

## How the others do this

- **LangGraph** (LangChain) passes a callback handler in the run's `config`; its `on_llm_end` sees every
  model call as it happens, nested agents included.
- **CrewAI** publishes an event on its event bus after every model call; a listener collects them, and
  the bus must be flushed before reading, because events arrive asynchronously.

All three feed the same `RunMetrics`, and all three count tool calls through the shared tools.

## Takeaways

- Fix the definitions once, in shared code, then connect each framework to them.
- Count what you can outside the framework (tool calls); take only model calls from the framework.
- In Strands, settle a hook's numbers when an invocation ends, not after each call.
- An eval is a fixed set of cases, scored automatically, re-run after every change.
- That completes the components. The capstones build ShipFast and the research agent in full.